# 02 — Preprocessing

Load the filtered lewtun/music_genres dataset, compute mel-spectrograms and MFCCs,
and save normalised tensors to `/kaggle/working/tensors/` for use by the CNN and LSTM pipelines.
The output folder should be uploaded as a Kaggle Dataset called **music-genre-tensors**
and shared with the group.

## 1. Imports and config

All dependencies, audio/feature constants, output directory, and the genre exclusion list
(identical to the EDA notebook so the two are consistent).

In [1]:
from pathlib import Path
import json

import numpy as np
import torch
import librosa
import skimage.transform
from datasets import load_dataset, concatenate_datasets
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Audio constants
SAMPLE_RATE    = 22050
DURATION       = 30
TARGET_SAMPLES = SAMPLE_RATE * DURATION
N_MELS         = 128
N_MFCC         = 40
HOP_LENGTH     = 512
N_FFT          = 2048
IMG_SIZE       = 128

OUTPUT_DIR = Path("/kaggle/working/tensors")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REMOVE_GENRES = {
    "Spoken", "Old-Time / Historic", "Ambient Electronic",
    "Unknown", "Easy Listening", "Blues", "Soul-RnB", "International"
}

print("Imports OK")
print(f"Output directory: {OUTPUT_DIR}")

Imports OK
Output directory: /kaggle/working/tensors


## 2. Load and filter dataset

Download from HuggingFace, concatenate the pre-split train and test shards into one unified
dataset, then drop the eight excluded genres. The filtered dataset is what all downstream
cells operate on.

In [2]:
ds_raw  = load_dataset("lewtun/music_genres")
ds_full = concatenate_datasets([ds_raw["train"], ds_raw["test"]])

ds = ds_full.filter(lambda ex: ex["genre"] not in REMOVE_GENRES)

genres = sorted(set(ds["genre"]))
print(f"Total samples after filtering : {len(ds):,}")
print(f"Genres ({len(genres)})           : {genres}")

README.md:   0%|          | 0.00/545 [00:00<?, ?B/s]

data/train-00000-of-00016-6b5481c76a3d27(…):   0%|          | 0.00/489M [00:00<?, ?B/s]

data/train-00001-of-00016-438bb9cb7b0600(…):   0%|          | 0.00/488M [00:00<?, ?B/s]

data/train-00002-of-00016-c1a616564aeae4(…):   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00003-of-00016-73a29e154975c4(…):   0%|          | 0.00/488M [00:00<?, ?B/s]

...

data/test-00003-of-00004-00213d3ef9894ab(…):   0%|          | 0.00/494M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19909 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5076 [00:00<?, ? examples/s]

Filter:   0%|          | 0/24985 [00:00<?, ? examples/s]

Total samples after filtering : 17,632
Genres (12)           : ['Chiptune / Glitch', 'Classical', 'Country', 'Electronic', 'Experimental', 'Folk', 'Hip-Hop', 'Instrumental', 'Jazz', 'Pop', 'Punk', 'Rock']


## 3. Stratified train / val / test split (70 / 15 / 15)

Split indices with stratification on genre label so every split has the same class
proportions. The genre-to-integer mapping is saved to `genre_mapping.json` so that
all downstream notebooks use an identical label encoding.

In [3]:
genre_mapping = {g: i for i, g in enumerate(genres)}
mapping_path  = OUTPUT_DIR / "genre_mapping.json"
with open(mapping_path, "w") as f:
    json.dump(genre_mapping, f, indent=2)
print(f"Saved genre mapping → {mapping_path}")

indices = list(range(len(ds)))
labels  = ds["genre"]

idx_train, idx_tmp = train_test_split(
    indices, test_size=0.30, stratify=labels, random_state=42
)
labels_tmp = [labels[i] for i in idx_tmp]
idx_val, idx_test = train_test_split(
    idx_tmp, test_size=0.50, stratify=labels_tmp, random_state=42
)

print(f"\nSplit sizes:")
print(f"  Train : {len(idx_train):,}")
print(f"  Val   : {len(idx_val):,}")
print(f"  Test  : {len(idx_test):,}")

# Confirm stratification: compare genre proportions across splits
def genre_dist(idx_list):
    counts = {}
    for i in idx_list:
        g = labels[i]
        counts[g] = counts.get(g, 0) + 1
    total = sum(counts.values())
    return {g: round(c / total * 100, 1) for g, c in sorted(counts.items())}

print("\nGenre proportions (%) — train:", genre_dist(idx_train))
print("Genre proportions (%) — val  :", genre_dist(idx_val))
print("Genre proportions (%) — test :", genre_dist(idx_test))

Saved genre mapping → /kaggle/working/tensors/genre_mapping.json

Split sizes:
  Train : 12,342
  Val   : 2,645
  Test  : 2,645

Genre proportions (%) — train: {'Chiptune / Glitch': 6.7, 'Classical': 2.8, 'Country': 0.8, 'Electronic': 17.4, 'Experimental': 10.2, 'Folk': 6.9, 'Hip-Hop': 10.0, 'Instrumental': 5.9, 'Jazz': 1.7, 'Pop': 5.4, 'Punk': 14.6, 'Rock': 17.5}
Genre proportions (%) — val  : {'Chiptune / Glitch': 6.7, 'Classical': 2.8, 'Country': 0.8, 'Electronic': 17.4, 'Experimental': 10.2, 'Folk': 6.9, 'Hip-Hop': 9.9, 'Instrumental': 5.9, 'Jazz': 1.7, 'Pop': 5.4, 'Punk': 14.7, 'Rock': 17.6}
Genre proportions (%) — test : {'Chiptune / Glitch': 6.7, 'Classical': 2.8, 'Country': 0.8, 'Electronic': 17.4, 'Experimental': 10.2, 'Folk': 6.9, 'Hip-Hop': 10.0, 'Instrumental': 5.9, 'Jazz': 1.7, 'Pop': 5.4, 'Punk': 14.6, 'Rock': 17.5}


## 4. Feature extraction functions

Two functions share the same audio normalisation pipeline (resample, mono, pad/trim).
`extract_melspectrogram` returns a 128 × 128 log-mel image for the CNN.
`extract_mfcc` returns a (frames × 40) matrix for the LSTM.

In [4]:
def _normalise_audio(audio_array: np.ndarray, sr: int) -> np.ndarray:
    """Resample, convert to mono, and pad/trim to TARGET_SAMPLES."""
    audio = np.array(audio_array, dtype=np.float32)
    if audio.ndim == 2:
        if audio.shape[0] <= 2:
            audio = audio.mean(axis=0)  # (channels, samples) → (samples,)
        else:
            audio = audio.mean(axis=1)  # (samples, channels) → (samples,)
    if sr != SAMPLE_RATE:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)
    if len(audio) < TARGET_SAMPLES:
        audio = np.pad(audio, (0, TARGET_SAMPLES - len(audio)))
    else:
        audio = audio[:TARGET_SAMPLES]
    return audio


def extract_melspectrogram(audio_array: np.ndarray, sr: int) -> np.ndarray:
    """Return a float32 log-mel spectrogram of shape (IMG_SIZE, IMG_SIZE)."""
    audio = _normalise_audio(audio_array, sr)
    mel   = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_mels=N_MELS,
        hop_length=HOP_LENGTH, n_fft=N_FFT
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_resized = skimage.transform.resize(
        mel_db, (IMG_SIZE, IMG_SIZE), anti_aliasing=True
    )
    return mel_resized.astype(np.float32)


def extract_mfcc(audio_array: np.ndarray, sr: int) -> np.ndarray:
    """Return a float32 MFCC matrix of shape (frames, N_MFCC)."""
    audio = _normalise_audio(audio_array, sr)
    mfcc  = librosa.feature.mfcc(
        y=audio, sr=SAMPLE_RATE, n_mfcc=N_MFCC, hop_length=HOP_LENGTH
    )
    return mfcc.T.astype(np.float32)  # (frames, coefficients)


print("Feature extraction functions defined.")

Feature extraction functions defined.


## 5. Process mel-spectrograms

Extract a 128 × 128 log-mel spectrogram for every clip. Normalisation statistics
(mean and std) are computed on the **train split only** and then applied to all
three splits to prevent data leakage. A channel dimension is added so the tensors
are ready for a 2D CNN: shape `(N, 1, 128, 128)`.

In [5]:
def process_spectrograms(idx_list, split_name):
    specs = []
    for i in tqdm(idx_list, desc=f"Spectrograms [{split_name}]"):
        try:
            ex    = ds[i]
            arr   = ex["audio"]["array"]
            sr    = ex["audio"]["sampling_rate"]
            specs.append(extract_melspectrogram(arr, sr))
        except Exception as e:
            raise RuntimeError(f"Preprocessing failed at index {i}") from e
    return np.stack(specs, axis=0)  # (N, 128, 128)


spec_train_raw = process_spectrograms(idx_train, "train")
spec_val_raw   = process_spectrograms(idx_val,   "val")
spec_test_raw  = process_spectrograms(idx_test,  "test")

# Normalise using train statistics only
spec_mean = float(spec_train_raw.mean())
spec_std  = float(spec_train_raw.std())
if spec_std == 0.0:
    spec_std = 1.0
print(f"\nSpectrogram stats (train) — mean: {spec_mean:.4f}, std: {spec_std:.4f}")

spec_train = (spec_train_raw - spec_mean) / spec_std
spec_val   = (spec_val_raw   - spec_mean) / spec_std
spec_test  = (spec_test_raw  - spec_mean) / spec_std

# Add channel dimension → (N, 1, 128, 128)
for split_name, arr, fname in [
    ("train", spec_train, "spectrograms_train.pt"),
    ("val",   spec_val,   "spectrograms_val.pt"),
    ("test",  spec_test,  "spectrograms_test.pt"),
]:
    t = torch.tensor(arr[:, np.newaxis, :, :], dtype=torch.float32)
    torch.save(t, OUTPUT_DIR / fname)
    print(f"Saved {fname} — shape {tuple(t.shape)}")

with open(OUTPUT_DIR / "spectrogram_stats.json", "w") as f:
    json.dump({"mean": spec_mean, "std": spec_std}, f, indent=2)
print(f"Saved spectrogram_stats.json")

Spectrograms [train]:   0%|          | 0/12342 [00:00<?, ?it/s]

Spectrograms [val]:   0%|          | 0/2645 [00:00<?, ?it/s]

Spectrograms [test]:   0%|          | 0/2645 [00:00<?, ?it/s]


Spectrogram stats (train) — mean: -42.8131, std: 15.0116
Saved spectrograms_train.pt — shape (12342, 1, 128, 128)
Saved spectrograms_val.pt — shape (2645, 1, 128, 128)
Saved spectrograms_test.pt — shape (2645, 1, 128, 128)
Saved spectrogram_stats.json


## 6. Process MFCCs

Extract 40 MFCC coefficients per frame for every clip. Sequences are resized using
temporal interpolation to a fixed length of 130 frames so the full 30-second clip is
represented, rather than padding or truncating. Normalisation is per-feature-dimension
(mean and std vectors of length 40), computed on the **train split only**.
Final shape: `(N, 130, 40)`.

In [6]:
MFCC_FRAMES = 130


def resize_mfcc_sequence(mfcc: np.ndarray) -> np.ndarray:
    """Resize MFCC sequence to exactly (MFCC_FRAMES, N_MFCC)
    using interpolation so the full 30s clip is represented."""
    return skimage.transform.resize(
        mfcc,
        (MFCC_FRAMES, N_MFCC),
        anti_aliasing=True
    ).astype(np.float32)


def process_mfccs(idx_list, split_name):
    mfccs = []
    for i in tqdm(idx_list, desc=f"MFCCs [{split_name}]"):
        try:
            ex    = ds[i]
            arr   = ex["audio"]["array"]
            sr    = ex["audio"]["sampling_rate"]
            m     = extract_mfcc(arr, sr)
            mfccs.append(resize_mfcc_sequence(m))
        except Exception as e:
            raise RuntimeError(f"Preprocessing failed at index {i}") from e
    return np.stack(mfccs, axis=0)  # (N, 130, 40)


mfcc_train_raw = process_mfccs(idx_train, "train")
mfcc_val_raw   = process_mfccs(idx_val,   "val")
mfcc_test_raw  = process_mfccs(idx_test,  "test")

# Per-feature normalisation using train statistics only
mfcc_mean = mfcc_train_raw.mean(axis=(0, 1))   # shape (40,)
mfcc_std  = mfcc_train_raw.std(axis=(0, 1))    # shape (40,)
mfcc_std  = np.where(mfcc_std == 0, 1.0, mfcc_std)  # avoid division by zero
print(f"MFCC mean shape: {mfcc_mean.shape}, std shape: {mfcc_std.shape}")

mfcc_train = (mfcc_train_raw - mfcc_mean) / mfcc_std
mfcc_val   = (mfcc_val_raw   - mfcc_mean) / mfcc_std
mfcc_test  = (mfcc_test_raw  - mfcc_mean) / mfcc_std

for split_name, arr, fname in [
    ("train", mfcc_train, "mfccs_train.pt"),
    ("val",   mfcc_val,   "mfccs_val.pt"),
    ("test",  mfcc_test,  "mfccs_test.pt"),
]:
    t = torch.tensor(arr, dtype=torch.float32)
    torch.save(t, OUTPUT_DIR / fname)
    print(f"Saved {fname} — shape {tuple(t.shape)}")

with open(OUTPUT_DIR / "mfcc_stats.json", "w") as f:
    json.dump(
        {"mean": mfcc_mean.tolist(), "std": mfcc_std.tolist()},
        f, indent=2
    )
print("Saved mfcc_stats.json")

MFCCs [train]:   0%|          | 0/12342 [00:00<?, ?it/s]

MFCCs [val]:   0%|          | 0/2645 [00:00<?, ?it/s]

MFCCs [test]:   0%|          | 0/2645 [00:00<?, ?it/s]

MFCC mean shape: (40,), std shape: (40,)
Saved mfccs_train.pt — shape (12342, 130, 40)
Saved mfccs_val.pt — shape (2645, 130, 40)
Saved mfccs_test.pt — shape (2645, 130, 40)
Saved mfcc_stats.json


## 7. Save labels

Integer-encode genre strings using the mapping saved in cell 3,
then write one `torch.long` tensor per split. Label tensors are
1-D with shape `(N,)` and align index-for-index with the spectrogram
and MFCC tensors saved above.

In [7]:
for split_name, idx_list, fname in [
    ("train", idx_train, "labels_train.pt"),
    ("val",   idx_val,   "labels_val.pt"),
    ("test",  idx_test,  "labels_test.pt"),
]:
    encoded = [genre_mapping[ds[i]["genre"]] for i in idx_list]
    t = torch.tensor(encoded, dtype=torch.long)
    torch.save(t, OUTPUT_DIR / fname)
    print(f"Saved {fname} — shape {tuple(t.shape)}")

Saved labels_train.pt — shape (12342,)
Saved labels_val.pt — shape (2645,)
Saved labels_test.pt — shape (2645,)


## 8. Verification

Reload every saved tensor from disk and confirm shapes match expectations.
Also prints the genre mapping and per-split class distribution to make it
easy to spot any encoding errors before the tensors are uploaded.

In [8]:
print("=" * 50)
print("TENSOR SHAPES")
print("=" * 50)

files_to_check = [
    ("spectrograms_train.pt", (None, 1, 128, 128)),
    ("spectrograms_val.pt",   (None, 1, 128, 128)),
    ("spectrograms_test.pt",  (None, 1, 128, 128)),
    ("mfccs_train.pt",        (None, 130, 40)),
    ("mfccs_val.pt",          (None, 130, 40)),
    ("mfccs_test.pt",         (None, 130, 40)),
    ("labels_train.pt",       (None,)),
    ("labels_val.pt",         (None,)),
    ("labels_test.pt",        (None,)),
]

for fname, expected in files_to_check:
    t = torch.load(OUTPUT_DIR / fname, weights_only=True)
    shape_ok = all(
        e is None or t.shape[i] == e
        for i, e in enumerate(expected)
    )
    status = "OK" if shape_ok else "MISMATCH"
    print(f"  [{status}] {fname:35s} {tuple(t.shape)}")

print()
print("GENRE MAPPING")
for g, idx in genre_mapping.items():
    print(f"  {idx:2d} — {g}")

print()
print("CLASS DISTRIBUTION PER SPLIT")
for split_name, fname in [("train", "labels_train.pt"), ("val", "labels_val.pt"), ("test", "labels_test.pt")]:
    t = torch.load(OUTPUT_DIR / fname, weights_only=True)
    unique, counts = t.unique(return_counts=True)
    inv_map = {v: k for k, v in genre_mapping.items()}
    dist = {inv_map[int(u)]: int(c) for u, c in zip(unique, counts)}
    print(f"  {split_name}: {dist}")

print()
print("ALIGNMENT CHECK")
for split in ["train", "val", "test"]:
    spec = torch.load(OUTPUT_DIR / f"spectrograms_{split}.pt", weights_only=True)
    mfcc = torch.load(OUTPUT_DIR / f"mfccs_{split}.pt", weights_only=True)
    lbls = torch.load(OUTPUT_DIR / f"labels_{split}.pt", weights_only=True)
    assert spec.shape[0] == lbls.shape[0], \
        f"Spectrogram/label mismatch in {split}"
    assert mfcc.shape[0] == lbls.shape[0], \
        f"MFCC/label mismatch in {split}"
    print(f"  [{split}] alignment OK — {lbls.shape[0]:,} samples")

TENSOR SHAPES
  [OK] spectrograms_train.pt               (12342, 1, 128, 128)
  [OK] spectrograms_val.pt                 (2645, 1, 128, 128)
  [OK] spectrograms_test.pt                (2645, 1, 128, 128)
  [OK] mfccs_train.pt                      (12342, 130, 40)
  [OK] mfccs_val.pt                        (2645, 130, 40)
  [OK] mfccs_test.pt                       (2645, 130, 40)
  [OK] labels_train.pt                     (12342,)
  [OK] labels_val.pt                       (2645,)
  [OK] labels_test.pt                      (2645,)

GENRE MAPPING
   0 — Chiptune / Glitch
   1 — Classical
   2 — Country
   3 — Electronic
   4 — Experimental
   5 — Folk
   6 — Hip-Hop
   7 — Instrumental
   8 — Jazz
   9 — Pop
  10 — Punk
  11 — Rock

CLASS DISTRIBUTION PER SPLIT
  train: {'Chiptune / Glitch': 827, 'Classical': 347, 'Country': 99, 'Electronic': 2150, 'Experimental': 1260, 'Folk': 850, 'Hip-Hop': 1230, 'Instrumental': 731, 'Jazz': 214, 'Pop': 661, 'Punk': 1807, 'Rock': 2166}
  val: {'Chiptu

## 9. Summary

List all output files with their sizes, then print the instructions for
uploading the tensors as a Kaggle Dataset so the CNN and LSTM notebooks
can consume them via `/kaggle/input/music-genre-tensors/`.

In [9]:
print("=" * 55)
print("OUTPUT FILES")
print("=" * 55)
total_mb = 0.0
for p in sorted(OUTPUT_DIR.iterdir()):
    mb = p.stat().st_size / 1_048_576
    total_mb += mb
    print(f"  {p.name:40s} {mb:7.2f} MB")
print(f"  {'TOTAL':40s} {total_mb:7.2f} MB")

print()
print("SPLIT SIZES")
print(f"  Train : {len(idx_train):,}")
print(f"  Val   : {len(idx_val):,}")
print(f"  Test  : {len(idx_test):,}")

print()
print(
    "Next step: download the /kaggle/working/tensors/ folder, "
    "create a new Kaggle Dataset called 'music-genre-tensors', "
    "upload all files, and share the dataset with your group. "
    "CNN and LSTM notebooks will read from "
    "/kaggle/input/music-genre-tensors/"
)

OUTPUT FILES
  genre_mapping.json                          0.00 MB
  labels_test.pt                              0.02 MB
  labels_train.pt                             0.10 MB
  labels_val.pt                               0.02 MB
  mfcc_stats.json                             0.00 MB
  mfccs_test.pt                              52.47 MB
  mfccs_train.pt                            244.82 MB
  mfccs_val.pt                               52.47 MB
  spectrogram_stats.json                      0.00 MB
  spectrograms_test.pt                      165.31 MB
  spectrograms_train.pt                     771.38 MB
  spectrograms_val.pt                       165.31 MB
  TOTAL                                    1451.91 MB

SPLIT SIZES
  Train : 12,342
  Val   : 2,645
  Test  : 2,645

Next step: download the /kaggle/working/tensors/ folder, create a new Kaggle Dataset called 'music-genre-tensors', upload all files, and share the dataset with your group. CNN and LSTM notebooks will read from /kaggle/inpu